# pitch-analytics · 01 — Player detection & tracking

Goal: detect players + ball in a football clip, give each player a stable ID, and export an annotated video.

**Before running:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
!pip install -q ultralytics supervision
!nvidia-smi --query-gpu=name --format=csv

## 1. Upload a clip
Use a short (30–60 s) wide-angle clip of a match — your own pickup game or a highlight.

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO = list(uploaded.keys())[0]
print('Using:', VIDEO)

## 2. Detect + track
Pretrained COCO model: class `0` = person, `32` = sports ball. ByteTrack assigns IDs.

In [ ]:
from ultralytics import YOLO
import supervision as sv
import numpy as np
from collections import defaultdict

model = YOLO("yolo11m.pt")          # downloads automatically
tracker = sv.ByteTrack()

box_ann   = sv.BoxAnnotator(thickness=2)
label_ann = sv.LabelAnnotator(text_scale=0.5)
trace_ann = sv.TraceAnnotator(trace_length=40)
ball_ann  = sv.TriangleAnnotator(base=20, height=17)

info = sv.VideoInfo.from_video_path(VIDEO)
MAX_SECONDS = 30                                   # keep first runs fast
frames = sv.get_video_frames_generator(VIDEO, end=int(info.fps * MAX_SECONDS))

positions = defaultdict(list)                      # tracker_id -> [(x, y), ...] in pixels

with sv.VideoSink("output.mp4", info) as sink:
    for frame in frames:
        result = model(frame, imgsz=1280, conf=0.25, verbose=False)[0]
        det = sv.Detections.from_ultralytics(result)

        ball    = det[det.class_id == 32]
        players = tracker.update_with_detections(det[det.class_id == 0])

        for tid, (x, y) in zip(players.tracker_id,
                               players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)):
            positions[tid].append((x, y))

        out = box_ann.annotate(frame.copy(), players)
        out = trace_ann.annotate(out, players)
        out = label_ann.annotate(out, players, labels=[f"#{t}" for t in players.tracker_id])
        out = ball_ann.annotate(out, ball)
        sink.write_frame(out)

print(f"Done. {len(positions)} unique player IDs seen.")

## 3. Quick sanity stats
If you see many more IDs than real players, tracking is losing people (ID switches) — that's the next thing to fix.

In [ ]:
import pandas as pd
stats = pd.DataFrame({
    "player_id": list(positions.keys()),
    "frames_tracked": [len(v) for v in positions.values()],
}).sort_values("frames_tracked", ascending=False)
stats["seconds_on_screen"] = (stats.frames_tracked / info.fps).round(1)
stats.head(20)

## 4. Pixel heat map (preview)
This is in *video pixels*, not real pitch metres yet — homography comes in notebook 02.

In [ ]:
import matplotlib.pyplot as plt
pid = stats.player_id.iloc[0]            # most-tracked player
xy = np.array(positions[pid])
plt.figure(figsize=(10, 5))
plt.hist2d(xy[:, 0], xy[:, 1], bins=40, range=[[0, info.width], [0, info.height]], cmap="hot")
plt.gca().invert_yaxis()
plt.title(f"Player #{pid} — pixel heat map")
plt.colorbar(); plt.show()

## 5. Download result
If it won't play in the browser, open it with VLC.

In [ ]:
files.download('output.mp4')